In [42]:
import time
from typing import Callable

from requests import get, Response


def fetch_content(url: str) -> Response:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:155.0) Gecko/20100101 Firefox/155.0'
    }
    response = get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to open url {url}: {response.status_code}, {response.text}")

    print(f"Fetched content with total size: {len(response.content)} bytes")

    return response

def batch_requests(urls: list[str], func: Callable[[Response], None], timeout_ms: float = 0.1):
    for url in urls:
        response = fetch_content(url)
        func(response)
        time.sleep(timeout_ms)

In [43]:
from abc import ABC, abstractclassmethod, abstractmethod
from dataclasses import dataclass

from bs4 import BeautifulSoup


@dataclass
class Page(ABC):
    title: str
    text: str
    full_content: BeautifulSoup
    response: Response
    url: str
    
    def __init__(self, response: Response):
        self.full_content = BeautifulSoup(response.content, "html.parser")
        self.title = self.full_content.find_all("meta", attrs={"property": "og:title"})[0]["content"]
        self.response = response
        self.url = response.url
        self.text = self.get_text_content()
    
    @abstractmethod
    def get_text_content(self) -> str:
        pass
    
    def get_total_size_in_bytes(self) -> int:
        return len(self.response.content)
    
    def get_info_size_in_bytes(self) -> int:
        return len(bytes(self.title + self.text, encoding="utf-8"))
    
    def save_to_file(self, path: str = './page.html') -> None:
        with open(path, 'w') as f:
            f.write(self.full_content.prettify())

@dataclass
class Source(ABC):
    name: str
    url: str
    sample_page: Page
    page_count: int
    
    @abstractclassmethod
    def count_pages(cls) -> int:
        pass

    def get_approximate_total_size_in_megabytes(self) -> int:
        return round(self.page_count * self.sample_page.get_total_size_in_bytes() / (1024 * 1024))

    def get_approximate_parsed_size_in_megabytes(self) -> int:
        return round(self.page_count * self.sample_page.get_info_size_in_bytes() / (1024 * 1024))

In [44]:
@dataclass
class RiaPage(Page):
    def __init__(self, response: Response):
        super().__init__(response)
    
    def get_text_content(self) -> str:
        content_block = self.full_content.find_all("div", attrs={"class": "article__body js-mediator-article mia-analytics"})[0]
        
        div_content_blocks = content_block.find_all("div", attrs={"class": ["article__block"]})
        text_content_blocks = [block for block in div_content_blocks if block["data-type"] == "text" or block["data-type"] == "quote"]
        
        text_blocks = [block.text for block in text_content_blocks]
        return "\n".join(text_blocks)


@dataclass
class RiaSource(Source):
    url = "https://ria.ru"
    
    def __init__(self):
        self.name = "Ria news"
        self.sample_page = self._get_sample_page()
        self.page_count = self.count_pages()

    @classmethod
    def _get_sample_page(cls) -> RiaPage:
        response = fetch_content(f"{cls.url}/20260916/ukraina-2117935384.html")
        return RiaPage(response)

    @classmethod
    def _get_article_group_links(cls) -> list[str]:
        response = fetch_content(f"{cls.url}/sitemap_article_index.xml")
        xml_content = BeautifulSoup(response.content, "xml")
        return [item.text for item in xml_content.find_all("loc")]

    @classmethod
    def _get_article_count_for_groups(cls, urls: list[str]) -> int:
        total_count = 0

        def parser(res: Response):
            nonlocal total_count

            xml_content = BeautifulSoup(res.content, "xml")
            found_links = len(xml_content.find_all("url"))
            
            print(f"Found {found_links} for ria source")
            
            total_count += found_links
            

        batch_requests(urls, parser)
        return total_count
    
    @classmethod
    def count_pages(cls) -> int:
        article_group_links = cls._get_article_group_links()
        return cls._get_article_count_for_groups(article_group_links)

In [45]:
ria = RiaSource()
ria.sample_page.title, ria.sample_page.text, ria.page_count

Fetched content with total size: 175594 bytes
Fetched content with total size: 35490 bytes
Fetched content with total size: 2242654 bytes
Found 13420 for ria source
Fetched content with total size: 3788653 bytes
Found 22556 for ria source
Fetched content with total size: 3744956 bytes
Found 22305 for ria source
Fetched content with total size: 3794759 bytes
Found 22593 for ria source
Fetched content with total size: 3627299 bytes
Found 21590 for ria source
Fetched content with total size: 3430878 bytes
Found 20440 for ria source
Fetched content with total size: 4395238 bytes
Found 26207 for ria source
Fetched content with total size: 3809190 bytes
Found 22668 for ria source
Fetched content with total size: 3826170 bytes
Found 22794 for ria source
Fetched content with total size: 4423274 bytes
Found 26222 for ria source
Fetched content with total size: 4219660 bytes
Found 24998 for ria source
Fetched content with total size: 3952347 bytes
Found 23388 for ria source
Fetched content with 

('Глава МИД Украины выдвинул наглое требование после удара ВС России',
 'МОСКВА, 16 сен — РИА Новости. Глава украинского МИД Андрей Сибига потребовал от союзников передачи Киеву локомотивов после российского удара в Волынской области.\n«"Мы призываем наших партнеров оказать неотложную помощь в поиске локомотивов, совместимых с колеей 1520 миллиметров, а также в финансировании ускоренной закупки новых единиц подвижного состава", — написал он в социальных сетях.\nВо вторник Минобороны сообщило, что Вооруженные силы России нанесли удар по железнодорожному составу в Рымачах Волынской области, возившему грузы военного назначения.\nОтмечается, что регулярные удары по объектам железнодорожной инфраструктуры и локомотивам, задействованным в снабжении войск, нарушают транспортную логистику ВСУ.',
 4453036)

In [46]:
ria.page_count

4453036

In [47]:
ria.get_approximate_total_size_in_megabytes(), ria.get_approximate_parsed_size_in_megabytes()

(745703, 6162)

In [51]:
ria.sample_page.save_to_file("ria.html")

In [48]:
@dataclass
class IzvestiaPage(Page):
    def __init__(self, response: Response):
        super().__init__(response)

    def get_text_content(self) -> str:
        content_block = self.full_content.find_all("div", attrs={"itemprop": "articleBody"})[0]

        text_content_blocks = [block for block in content_block.find_all("p")]
        text_blocks = [block.text for block in text_content_blocks]

        return "\n".join(text_blocks)

@dataclass
class IzvestiaSource(Source):
    url = "https://iz.ru"

    def __init__(self):
        self.name = "Tass news"
        self.sample_page = self._get_sample_page()
        self.page_count = self.count_pages()

    @classmethod
    def _get_sample_page(cls) -> IzvestiaPage:
        response = fetch_content(f"{cls.url}/2168313/2026-09-16/frs-ssha-povysila-kliuchevuiu-stavku-vpervye-za-tri-goda")
        return IzvestiaPage(response)

    @classmethod
    def _get_article_group_links(cls) -> list[str]:
        response = fetch_content(f"{cls.url}/sitemap.xml")
        xml_content = BeautifulSoup(response.content, "xml")
        return [item.text for item in xml_content.find_all("loc")]

    @classmethod
    def _get_article_count_for_groups(cls, urls: list[str]) -> int:
        total_count = 0

        def parser(res: Response):
            nonlocal total_count

            xml_content = BeautifulSoup(res.content, "xml")
            found_links = len(xml_content.find_all("url"))

            print(f"Found {found_links} for tass source")

            total_count += found_links
        
        batch_requests(urls, parser)
        return total_count

    @classmethod
    def count_pages(cls) -> int:
        article_group_links = cls._get_article_group_links()
        return cls._get_article_count_for_groups(article_group_links)


In [49]:
izvestia_source = IzvestiaSource()
izvestia_source.sample_page.title, izvestia_source.sample_page.text

Fetched content with total size: 173108 bytes
Fetched content with total size: 6405 bytes
Fetched content with total size: 673906 bytes
Found 2999 for tass source
Fetched content with total size: 3824473 bytes
Found 25000 for tass source
Fetched content with total size: 3821326 bytes
Found 25000 for tass source
Fetched content with total size: 3823380 bytes
Found 25000 for tass source
Fetched content with total size: 3835501 bytes
Found 25000 for tass source
Fetched content with total size: 3833999 bytes
Found 25000 for tass source
Fetched content with total size: 3845592 bytes
Found 25000 for tass source
Fetched content with total size: 3828645 bytes
Found 25000 for tass source
Fetched content with total size: 3819166 bytes
Found 25000 for tass source
Fetched content with total size: 3819212 bytes
Found 25000 for tass source
Fetched content with total size: 3830787 bytes
Found 24999 for tass source
Fetched content with total size: 3975082 bytes
Found 25000 for tass source
Fetched cont

('ФРС США повысила ключевую ставку впервые за три года',
 'Федеральная резервная система (ФРС) США повысила процентную ставку по федеральным кредитным средствам на 25 базисных пунктов впервые с 2023 года. Об этом 16 сентября сообщили на сайте ФРС.\nТеперь диапазон ставки составляет 3,75–4% годовых. Решение американского регулятора совпало с прогнозами большинства аналитиков и участников рынка. За ужесточение денежно-кредитной политики проголосовали все 12 членов FOMC.\nПовышению предшествовал период смягчения политики: в 2024 году ФРС трижды снижала ставку, еще три раунда снижения состоялись в 2025 году.\nПрезидент США Дональд Трамп 13 сентября заявил, что Вашингтон прекратит торговлю с некоторыми странами, если Федеральная резервная система не снизит базовую процентную ставку. Он также указал, что предыдущие президенты США допустили «огромный дефицит», который повлек потерю средств на импорте.')

In [50]:
izvestia_source.page_count, izvestia_source.get_approximate_parsed_size_in_megabytes(), izvestia_source.get_approximate_total_size_in_megabytes()

(1613179, 2505, 266318)

In [52]:
izvestia_source.sample_page.save_to_file("izvestia.html")